# Lab 8: Approximate Q-learning and policy gradient

The first part learns to play a game of falling blocks with Q-learning, where the table of
Lab 7 is replaced by a linear function of features of the board. The second part learns to
balance a pole on a cart with policy gradient methods: REINFORCE, the effect of a baseline,
and an actor-critic method.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

In [ ]:
# Checks that your folder seqlearn/ is recent enough, and updates it otherwise. (Provided, just run it.)
import importlib.util, io, pathlib, re, shutil, sys, urllib.request, zipfile

REQUIRED = "2026.11.03"
URL = "https://igorcolin.github.io/teaching/sequential-decision/sequential-decision-setup.zip"


def seqlearn_version():
    spec = importlib.util.find_spec("seqlearn")
    if spec is None or spec.origin is None:
        return None, pathlib.Path("seqlearn")
    init = pathlib.Path(spec.origin)
    match = re.search(r'^__version__ = "([^"]+)"', init.read_text(), re.MULTILINE)
    return (match.group(1) if match else "none"), init.parent


version, folder = seqlearn_version()
if version is not None and version != "none" and version >= REQUIRED:
    print(f"seqlearn {version}: up to date")
else:
    print(f"seqlearn {version or 'not found'}: version {REQUIRED} or later needed, downloading it")
    try:
        data = urllib.request.urlopen(URL, timeout=30).read()
    except OSError as error:
        raise RuntimeError(f"download failed ({error}): download {URL} by hand, unzip it, "
                           "and replace your folder seqlearn/ with the new one") from None
    old = folder.with_name("seqlearn.old")
    if folder.exists():
        shutil.rmtree(old, ignore_errors=True)
        folder.rename(old)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            parts = pathlib.PurePosixPath(name).parts
            if len(parts) > 2 and parts[1] == "seqlearn" and not name.endswith("/"):
                target = folder.joinpath(*parts[2:])
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    importlib.invalidate_caches()
    version = seqlearn_version()[0]
    if version == "none" or version < REQUIRED:
        raise RuntimeError(f"the course page still has seqlearn {version}: tell your teacher")
    print(f"seqlearn updated to {version} (previous folder kept as {old})")
    if "seqlearn" in sys.modules:
        print("Restart the kernel (Kernel > Restart), then run the notebook from the top.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import checks
from seqlearn.blocks import Blocks, placement_features
from seqlearn.control import CartPole, quadratic_features
from seqlearn.plotting import COLORS, plot_mean, set_style

set_style()

# Part I: Approximate Q-learning

## 1. Falling blocks

Pieces fall one at a time in a well of $6$ columns and $10$ rows. The state is the board and
the falling piece; an action is a placement, that is, a rotation and a column, after which
the piece drops straight down. Each full row disappears and gives a reward $1$. The game
ends when a piece sticks out of the top of the well, with a reward $-1$, or after $200$
pieces.

`game.reset()` returns the state `(board, piece)`, `game.actions(piece)` the list of the
legal placements of a piece, and `game.step(action)` returns `(state, reward, done)`, as in
Lab 7; `game.rows` counts the rows removed since the start of the game. The pieces are given as drawings, and can be changed: try
`Blocks(pieces=[["##", "##"], ["#..", "###"]])`.

In [ ]:
game = Blocks(max_pieces=200, rng=0)
board, piece = game.reset()
print(f"{len(game.pieces)} pieces, placements of the current piece: {game.actions(piece)}")
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for k, ax in enumerate(axes):
    game.draw(ax, board, piece, title=f"after {5 * k} pieces")
    for _ in range(5):
        actions = game.actions(piece)
        (board, piece), _, done = game.step(actions[rng.integers(len(actions))])
plt.show()

There are up to $2^{60} \approx 10^{18}$ boards, far too many for a table of Q-values: most
boards are never seen twice. Instead, $Q$ is a linear function of features:
$$Q_w(s, a) = w^\top \phi(s, a) \enspace,$$
where $\phi(s, a)$ describes the board obtained after the placement $a$ in the state $s$:
a constant $1$, the height of each column, the number of holes (empty cells below a filled
one), the bumpiness (sum of the height differences of neighboring columns) and the maximal
height, divided by the height of the well, and the number of rows removed. A placement that
ends the game has value $-1$, its reward. `placement_features(game, board, piece)` returns,
for all the legal placements at once, the list `actions`, the array `phi` whose row $k$ is
$\phi(s, a_k)$, the array of the rewards, and a boolean array `lost`, true for the
placements that end the game (their row of `phi` is zero).

In [ ]:
board, piece = game.reset()
actions, phi, rewards, lost = placement_features(game, board, piece)
print("first placement:", actions[0], " phi =", phi[0].round(2), " reward =", rewards[0])

## 2. Linear Q-learning

After a transition $(s, a, r, s')$, Q-learning moves $Q_w(s, a)$ toward the target
$y = r + \gamma \max_{a'} Q_w(s', a')$ ($y = r$ if the game is over). With a table, it moves
one entry. Here it takes a gradient step on $\frac12 \big(y - Q_w(s, a)\big)^2$, the target
being held fixed:
$$w \leftarrow w + \alpha \big(y - w^\top \phi(s, a)\big)\, \phi(s, a) \enspace.$$
This is *semi-gradient* Q-learning: the target also depends on $w$, but it is not
differentiated. If $\phi(s, a)$ is the indicator vector of the pair $(s, a)$, this is
exactly the update of Lab 7.

**Question 1.** Complete `LinearQAgent`: `q_values(board, piece)` returns the legal
placements, their features and the array of their Q-values ($-1$ for a placement that ends
the game); `act(board, piece)` plays $\varepsilon$-greedy and returns the placement and its
features (`None` if it ends the game); `update(phi, r, next_board, next_piece, done)`
applies the update above.

In [ ]:
class LinearQAgent:
    def __init__(self, game, n_features=11, alpha=0.01, gamma=0.95, epsilon=0.01, rng=None):
        self.game, self.alpha, self.gamma, self.epsilon = game, alpha, gamma, epsilon
        self.rng = np.random.default_rng(rng)
        self.w = np.zeros(n_features)

    def q_values(self, board, piece):
        actions, phi, rewards, lost = placement_features(self.game, board, piece)
        ...  # TODO
        return actions, phi, lost, q

    def act(self, board, piece):
        actions, phi, lost, q = self.q_values(board, piece)
        ...  # TODO
        return actions[k], (None if lost[k] else phi[k])

    def update(self, phi, r, next_board, next_piece, done):
        if phi is None:  # the placement ended the game: its value is its reward, -1
            return
        ...  # TODO


checks.check_linear_q(LinearQAgent)

The function below plays games and returns the number of rows removed in each of them.

In [ ]:
def play(game, agent, n_games, learn=True):
    rows = np.zeros(n_games)
    for k in range(n_games):
        (board, piece), done = game.reset(), False
        while not done:
            action, phi = agent.act(board, piece)
            (board, piece), r, done = game.step(action)
            if learn:
                agent.update(phi, r, board, piece, done)
        rows[k] = game.rows
    return rows

**Question 2.** Train $3$ agents (seeds $0$ to $2$, game `Blocks(max_pieces=200, rng=seed)`)
for $300$ games each with $\alpha = 0.01$, $\gamma = 0.95$ and $\varepsilon = 0.01$ (about
a minute), and plot the mean number of rows per game, with `plot_mean`. Compare with an
agent that plays at random ($\varepsilon = 1$, no learning). Print the weights of one agent:
which features does it penalize?

In [ ]:
...  # TODO

**Question 3.** Draw the board of a game played by a trained agent ($\varepsilon = 0$, no
learning) after $20$, $40$, $60$ and $80$ pieces.

In [ ]:
...  # TODO

Deep Q-networks (DQN) replace $w^\top \phi(s, a)$ with a neural network that reads the raw
board, so that the features are learned too. They add a replay buffer and a target network
to make the semi-gradient updates stable (see the lecture). The optional notebook
`lab08-torch` trains one on this game, with PyTorch.

# Part II: Policy gradient

## 3. The cart-pole

A pole is hinged on a cart that moves along a rail. The state is
$s = (x, \dot x, \theta, \dot\theta)$: the position and speed of the cart, the angle of the
pole ($0$ is upright) and its angular speed. Action $0$ pushes the cart to the left, action
$1$ to the right. Each step gives a reward $1$, and the episode ends when the pole leans by
more than $12$ degrees, when the cart leaves the rail, or after $500$ steps. The return of
an episode is therefore its length. `env.reset()` and `env.step(a)` work as before.

In [ ]:
env = CartPole(rng=0)
state, done, states = env.reset(), False, []
while not done:
    states.append(state)
    state, _, done = env.step(int(rng.integers(2)))
fig, axes = plt.subplots(1, 3, figsize=(10, 2))
for ax, t in zip(axes, [0, len(states) // 2, len(states) - 1]):
    env.draw(ax, states[t], title=f"step {t}")
plt.show()
print(f"random actions: the pole falls after {len(states)} steps")

## 4. Softmax policy

Instead of learning values, policy gradient methods learn the policy directly. Here
$$\pi_W(a \mid s) = \frac{e^{W_a^\top s}}{\sum_b e^{W_b^\top s}} \enspace,$$
where $W$ is a $2 \times 4$ matrix with rows $W_0, W_1$. As in Tutorial 8, exercise 1,
$\nabla_W \log \pi_W(a \mid s) = (e_a - \pi_W(\cdot \mid s))\, s^\top$, where $e_a$ is the
indicator vector of $a$ and $\pi_W(\cdot \mid s)$ the vector of the two probabilities.

**Question 4.** Complete `policy(W, s)`, which returns the vector of probabilities (subtract
the maximum of the logits before the exponential, as in Lab 5), and `grad_log_policy(W, s, a)`.

In [ ]:
def policy(W, s):
    ...  # TODO


def grad_log_policy(W, s, a):
    ...  # TODO


checks.check_policy(policy, grad_log_policy)

The function below plays an episode with the policy $\pi_W$ and returns the arrays of the
states, actions and rewards.

In [ ]:
def run_episode(env, W, rng):
    states, actions, rewards = [], [], []
    s, done = env.reset(), False
    while not done:
        a = int(rng.choice(2, p=policy(W, s)))
        states.append(s)
        actions.append(a)
        s, r, done = env.step(a)
        rewards.append(r)
    return np.array(states), np.array(actions), np.array(rewards)

## 5. REINFORCE

The objective is $J(W) = \mathbb E_{\pi_W}\big[\sum_t \gamma^t r_t\big]$. By the policy
gradient theorem (see the lecture),
$$\nabla J(W) \approx \mathbb E\Big[\sum_t \big(G_t - b\big)\, \nabla_W \log \pi_W(A_t \mid S_t)\Big]
\enspace, \qquad G_t = \sum_{u \ge t} \gamma^{u - t} r_u \enspace,$$
where $G_t$ is the return from step $t$ (the return-to-go) and $b$ a baseline, which does not
change the expectation (Tutorial 8, exercise 1). REINFORCE estimates this expectation with
the episodes it plays, and takes a gradient ascent step.

**Question 5.** Complete `returns_to_go(rewards, gamma)`, in one backward pass
($G_t = r_t + \gamma G_{t+1}$), and `reinforce_gradient(W, states, actions, rewards, gamma, b)`,
which returns $\sum_t (G_t - b) \nabla_W \log \pi_W(A_t \mid S_t)$ for one episode.

In [ ]:
def returns_to_go(rewards, gamma):
    ...  # TODO


def reinforce_gradient(W, states, actions, rewards, gamma, b=0.0):
    ...  # TODO


checks.check_returns_to_go(returns_to_go)
checks.check_reinforce_gradient(reinforce_gradient)

**Question 6.** Fix $W = 0$ (uniform policy) and $\gamma = 0.99$. Play $100$ preliminary
episodes, and let $\bar G$ be the mean of all their returns-to-go $G_t$ and $\bar G_0$ the
mean of their returns $G_0$. Then estimate the gradient from $2000$ episodes with
$b \in \{0, \bar G, \bar G_0\}$: compare the means of the estimates (first row of the
gradient, with $\pm 2$ standard errors) and the sums of the variances of their $8$
coordinates.

In [ ]:
...  # TODO

In [ ]:
...  # TODO

## 6. Actor-critic

An actor-critic method learns, next to the policy (the actor), an estimate
$V_v(s) = v^\top \psi(s)$ of the value of the states (the critic), where $\psi(s)$ is given
by `quadratic_features(s)` ($15$ features). After each transition $(s, a, r, s')$, it computes
the temporal difference error
$$\delta = r + \gamma V_v(s') - V_v(s) \enspace,$$
($\delta = r - V_v(s)$ if the pole fell; when the episode stops after $500$ steps, the pole
has not fallen, and $V_v(s')$ is kept), and updates both
$$v \leftarrow v + \alpha_v\, \delta\, \psi(s) \enspace, \qquad
W \leftarrow W + \alpha_W\, \delta\, \nabla_W \log \pi_W(a \mid s) \enspace.$$
$\delta$ estimates the advantage of $a$ in $s$: it replaces $G_t - b$, with a lower variance
(one reward instead of all the following ones), at the price of a bias while $V_v$ is wrong.

**Question 8.** Complete `ActorCritic`.

In [ ]:
class ActorCritic:
    def __init__(self, alpha_w=0.01, alpha_v=0.05, gamma=GAMMA, rng=None):
        self.alpha_w, self.alpha_v, self.gamma = alpha_w, alpha_v, gamma
        self.rng = np.random.default_rng(rng)
        self.W = np.zeros((2, 4))
        self.v = np.zeros(15)

    def act(self, s):
        return int(self.rng.choice(2, p=policy(self.W, s)))

    def update(self, s, a, r, s_next, failed):
        ...  # TODO


checks.check_actor_critic(ActorCritic)

**Question 9.** Train $5$ actor-critic agents (seeds $0$ to $4$) for $300$ episodes (about
one minute), and add their mean length of episode to the figure of Question 7.

In [ ]:
...  # TODO

## Going further

**Question 10.** Change the pieces of the falling blocks (for instance, add the square
`["##", "##"]` and the pieces of four cells of your choice) and train linear Q-learning
again. Is the game harder? Which weights change?

**Question 11.** In the actor-critic, add an entropy bonus: replace $\delta \nabla_W \log
\pi_W(a \mid s)$ with $\delta \nabla_W \log \pi_W(a \mid s) + \beta \nabla_W H(\pi_W(\cdot
\mid s))$, where $H(p) = -\sum_a p_a \log p_a$, for $\beta \in \{0, 0.01, 0.1\}$. Show that
$\nabla_W H(\pi_W(\cdot \mid s)) = -\sum_a \pi_W(a \mid s)\big(\log \pi_W(a \mid s) + H\big)
\, (e_a - \pi_W(\cdot \mid s))\, s^\top$, and compare the learning curves.

In [ ]:
...  # TODO